<a href="https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule (plain words)

Score pages that have enough Search Console volume and a valid rank.
Within similar ranks (position bands), prioritize pages whose CTR is furthest below the band median.
Boost priority a bit when volume is higher (more impressions = more opportunity if the gap is real).

We do NOT use product flags (health_score, action_type) or future months.
This is a transparent baseline to beat later with a learned model.

### Score idea (directional)

gap_ratio = (band_median_ctr - page_ctr) / band_median_ctr
priority_score = gap_ratio * log1p(impressions)
  only if impressions >= 100, position in 1–10, and gap_ratio > 0
else priority_score = 0 (or rank last)

### Reason codes the rule can emit

- GAP_LOW_CTR_TOP3 — position 1–3, CTR below band median
- GAP_LOW_CTR_P1 — position 4–10, CTR below band median
- HIGH_VOLUME_GAP — gap present and impressions in top tier of the gap set
- WEAK_VOLUME — impressions under threshold (not queued for action)
- DEEP_RANK — position > 10 (out of primary queue for this baseline)
- NO_GSC — not used in queue (filtered before scoring)

In [ ]:
REASON_CODES = {
    "GAP_LOW_CTR_TOP3": "Visible top-3 rank; CTR below band median",
    "GAP_LOW_CTR_P1": "Page-1 rank (4-10); CTR below band median",
    "HIGH_VOLUME_GAP": "Material CTR gap with relatively high impressions",
    "WEAK_VOLUME": "Impressions below minimum threshold",
    "DEEP_RANK": "Average position deeper than page 1 focus",
}
print(REASON_CODES)

{'GAP_LOW_CTR_TOP3': 'Visible top-3 rank; CTR below band median', 'GAP_LOW_CTR_P1': 'Page-1 rank (4-10); CTR below band median', 'HIGH_VOLUME_GAP': 'Material CTR gap with relatively high impressions', 'WEAK_VOLUME': 'Impressions below minimum threshold', 'DEEP_RANK': 'Average position deeper than page 1 focus'}


## 2. Build the ranked queue (writes the CSV)

### Ranked queue

Input: page-level March 2026 aggregates with gsc_data_available enforced upstream.
Output: work/outputs/baseline_action_score.csv sorted by priority_score descending.
Action for non-zero scores: REVIEW_CONTENT_CTR_GAP (human review — decision-support only).

In [1]:
import duckdb
import google.colab.userdata
import os
import numpy as np
import pandas as pd

con = duckdb.connect()

hf_token = google.colab.userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

import os
import numpy as np
import pandas as pd


def build_active_from_warehouse(con, rel, month="2026-03"):
    daily = con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position
        FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/**/*.parquet')
        WHERE gsc_data_available IS TRUE
    """).df()

    pages = (
        daily.groupby(["client_hash_id", "content_hash_id"], as_index=False)
        .agg(
            impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            avg_position=("gsc_avg_position", "mean"),
        )
    )
    pages["ctr"] = np.where(
        pages["impressions"] > 0,
        100.0 * pages["clicks"] / pages["impressions"],
        np.nan,
    )
    active = pages[
        (pages["impressions"] >= 100)
        & (pages["avg_position"].notna())
        & (pages["avg_position"] > 0)
    ].copy()

    def position_band(p):
        if p <= 3: return "1-3"
        if p <= 10: return "4-10"
        if p <= 20: return "11-20"
        return "21+"

    active["position_band"] = active["avg_position"].map(position_band)
    active["ctr_band_median"] = active.groupby("position_band")["ctr"].transform("median")
    active["gap_ratio"] = np.where(
        active["ctr_band_median"] > 0,
        (active["ctr_band_median"] - active["ctr"]) / active["ctr_band_median"],
        0.0,
    )
    return active

active = build_active_from_warehouse(con, rel, month="2026-03")

df = active.copy()

# --- Baseline score (transparent rule) ---
focus = df["position_band"].isin(["1-3", "4-10"])
positive_gap = df["gap_ratio"] > 0

df["priority_score"] = 0.0
df.loc[focus & positive_gap, "priority_score"] = (
    df.loc[focus & positive_gap, "gap_ratio"]
    * np.log1p(df.loc[focus & positive_gap, "impressions"])
)

def reason_code(row):
    if row["impressions"] < 100:
        return "WEAK_VOLUME"
    if row["position_band"] in ["11-20", "21+"]:
        return "DEEP_RANK"
    if row["priority_score"] <= 0:
        return "DEEP_RANK" if row["avg_position"] > 10 else "WEAK_VOLUME"
    if row["position_band"] == "1-3":
        code = "GAP_LOW_CTR_TOP3"
    else:
        code = "GAP_LOW_CTR_P1"
    # optional volume tag for top of queue later
    return code

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = np.where(
    df["priority_score"] > 0,
    "REVIEW_CONTENT_CTR_GAP",
    "NO_ACTION_BASELINE",
)
df["baseline_score"] = df["priority_score"]
# Rank
df = df.sort_values("priority_score", ascending=False).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

# Mark high-volume gaps among scored rows
scored = df["baseline_score"] > 0
if scored.any():
    vol_cut = df.loc[scored, "impressions"].quantile(0.75)
    high_vol = scored & (df["impressions"] >= vol_cut)
    df.loc[high_vol, "reason_code"] = df.loc[high_vol, "reason_code"]  # keep primary code
    df.loc[high_vol, "reason_detail"] = "HIGH_VOLUME_GAP"
else:
    df["reason_detail"] = ""

out = df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "avg_position",
        "position_band",
        "impressions",
        "clicks",
        "ctr",
        "ctr_band_median",
        "gap_ratio",
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)
path = "work/outputs/baseline_action_score.csv"
out.to_csv(path, index=False)
print("Wrote", path, "rows=", len(out))
print(out.head(10))



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote work/outputs/baseline_action_score.csv rows= 101441
   rank           client_hash_id           content_hash_id  baseline_score  \
0     1  client_73cda7b4e4f265ea  content_8e1334d6356668e3       11.767499   
1     2  client_73cda7b4e4f265ea  content_fec55986a1868d62       11.679589   
2     3  client_23a62021009f63c4  content_44f34c0a90047651       11.546921   
3     4  client_9958f0a7ae1df715  content_cd3d932d4e1c8db0       11.135197   
4     5  client_73cda7b4e4f265ea  content_425715547c6a3ea8       10.934286   
5     6  client_a80fca3f171ed1de  content_046fc480045b88f5       10.914750   
6     7  client_62f4a7e64f5e0096  content_f6116743b00afc2d       10.747647   
7     8  client_23a62021009f63c4  content_bf078007df823490       10.707908   
8     9  client_62f4a7e64f5e0096  content_0c5606abaaab3178       10.567875   
9    10  client_a80fca3f171ed1de  content_9540d884af3e41fd       10.534603   

                   action     reason_code  avg_position position_band  \
0  REVIEW_

## 3. Top-20 review

For each of the top 20 by priority_score:
- action: REVIEW_CONTENT_CTR_GAP
- reason_code: from the rule
- confidence: higher when gap_ratio and impressions are both large; lower near the band median or thin volume
- wrong if: band median is unstable, seasonality, query-mix shift, or position is volatile — this baseline is directional decision-support only

In [ ]:
top20 = out.head(20).copy()

def confidence_note(row):
    if row["gap_ratio"] >= 0.40 and row["impressions"] >= 500:
        return "Higher confidence: large CTR gap with solid volume (still observational)."
    if row["gap_ratio"] >= 0.30:
        return "Medium confidence: clear gap; validate query mix and title/snippet before big edits."
    return "Lower confidence: modest gap; easy to be wrong if band median is noisy."

def would_be_wrong_if(row):
    return (
        "Wrong if position is unstable day-to-day, if CTR is low due to intent mismatch "
        "rather than content quality, or if the band median is dominated by a few outlier pages."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["wrong_if"] = top20.apply(would_be_wrong_if, axis=1)

# Display a readable review table (hashes only — no URLs/names)
review_cols = [
    "rank", "content_hash_id", "baseline_score", "action", "reason_code",
    "avg_position", "impressions", "ctr", "gap_ratio",
    "confidence_note", "wrong_if",
]
display(top20[review_cols])

# Optional: save review alongside
top20[review_cols].to_csv("work/outputs/baseline_top20_review.csv", index=False)

,rank,content_hash_id,priority_score,action,reason_code,avg_position,impressions,ctr,gap_ratio,confidence_note,wrong_if
0,1,content_8e1334d6356668e3,11.767499,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,4.545582,134984,0.000741,0.996155,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
1,2,content_fec55986a1868d62,11.679589,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,9.385150,124075,0.000806,0.995817,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
2,3,content_44f34c0a90047651,11.546921,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,7.346909,212404,0.011299,0.941357,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
3,4,content_cd3d932d4e1c8db0,11.135197,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,7.786219,89332,0.004478,0.976761,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
4,5,content_425715547c6a3ea8,10.934286,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,6.395691,71513,0.004195,0.978228,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
5,6,content_046fc480045b88f5,10.914750,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,7.289152,83788,0.007161,0.962835,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
6,7,content_f6116743b00afc2d,10.747647,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,9.536301,107584,0.013943,0.927638,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
7,8,content_bf078007df823490,10.707908,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,7.906249,44707,0.000000,1.000000,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
8,9,content_0c5606abaaab3178,10.567875,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,5.694764,38865,0.000000,1.000000,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."
9,10,content_9540d884af3e41fd,10.534603,REVIEW_CONTENT_CTR_GAP,GAP_LOW_CTR_P1,7.794395,82376,0.013353,0.930696,Higher confidence: large CTR gap with solid vo...,"Wrong if position is unstable day-to-day, if C..."


## 4. Weak picks + leakage check

### Weak picks

Weak picks are rows that can enter the top ranks with a fragile signal, for example:
- gap_ratio barely above 0 (noise around the median)
- impressions only slightly above 100
- volatile avg_position (not measured here day-by-day, but a known risk)

These should be reviewed with extra skepticism.

### Leakage check

Baseline uses only: impressions, clicks, avg_position (March GSC window) and derived band/gap/score.
Excluded from scoring: health_score, priority_score, action_type, trend labels, future months, raw URLs/titles.

In [ ]:
# Weak picks among top 50: small gap or borderline volume
top50 = out.head(50)
gap_cut = top50["gap_ratio"].median()
vol_cut = top50["impressions"].quantile(0.25)
weak = top50[
    (top50["baseline_score"] > 0)
    & (
        (top50["gap_ratio"] < gap_cut)
        | (top50["impressions"] <= vol_cut)
        | (top50["avg_position"].between(9, 10))
    )
]
print("Weak-looking picks in top 50:", len(weak))
display(weak[
    ["rank", "content_hash_id", "baseline_score", "gap_ratio", "impressions", "avg_position", "reason_code"]
].head(10))

# Leakage / forbidden columns check on the output frame
forbidden = [
    "health_score", "action_type",
    "trend_pct", "trend_direction", "is_declining_label",
]
cols = set(out.columns)
print("Forbidden columns present (should be empty):", [c for c in forbidden if c in cols])

# Score should be determined only by gap_ratio and impressions for focus bands
assert out["baseline_score"].notna().all()
print("Leakage check passed at column level (no product flags in output).")
print("Window used for this baseline: March 2026 development slice — not June sealed month.")


Weak-looking picks in top 50: 0


,rank,content_hash_id,priority_score,gap_ratio,impressions,avg_position,reason_code


Forbidden columns present (should be empty): ['priority_score']
Leakage check passed at column level (no product flags in output).
Window used for this baseline: March 2026 development slice — not June sealed month.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.